In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt


In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Paths
# ============================================================

INPUT_PATH = Path("../data/processed/model_input/neutral_scoring_input.tsv")
MASTER_PATH = Path("../data/processed/master_snv_table.tsv")
RAW_DIR = Path("../data/raw")

OUTPUT_DIR = Path("../results/model")
FIGURE_BASE_DIR = Path("../results/figures")
SENSITIVITY_FIGURE_DIR = FIGURE_BASE_DIR / "model_threshold_sensitivity"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_BASE_DIR.mkdir(parents=True, exist_ok=True)
SENSITIVITY_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PHYLOP100WAY_PATH = RAW_DIR / "phyloP100way.tsv"

# ============================================================
# Final model configuration
# ============================================================

FINAL_FEATURE_SET_NAME = "codon_position_plus_phyloP100way"
FINAL_MODEL_NAME = f"{FINAL_FEATURE_SET_NAME}__isolation_forest"

# One model, three interpretation thresholds.
# T95 remains the main threshold used by downstream notebooks.
THRESHOLD_CONFIG = {
    "T90": 0.90,
    "T95": 0.95,
    "T99": 0.99,
}

THRESHOLD_NAMES = list(THRESHOLD_CONFIG.keys())
MAIN_THRESHOLD_NAME = "T95"

FIGURE_DIRS = {
    threshold_name: FIGURE_BASE_DIR / f"model_{threshold_name}"
    for threshold_name in THRESHOLD_NAMES
}

for figure_dir in FIGURE_DIRS.values():
    figure_dir.mkdir(parents=True, exist_ok=True)

SCORED_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"isolation_forest_scores_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

THRESHOLD_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"isolation_forest_threshold_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

ENRICHMENT_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"posthoc_enrichment_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

CLASSIFICATION_COUNTS_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"classification_counts_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

CLASSIFICATION_PRIMARY_GROUP_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"classification_primary_group_counts_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

TOP_OUTLIERS_OUTPUTS = {
    threshold_name: OUTPUT_DIR / f"top_outliers_{threshold_name}.tsv"
    for threshold_name in THRESHOLD_NAMES
}

# Combined QC outputs across all thresholds.
SCORED_ALL_OUTPUT = OUTPUT_DIR / "isolation_forest_scores_all_thresholds.tsv"
THRESHOLDS_ALL_OUTPUT = OUTPUT_DIR / "isolation_forest_thresholds_all.tsv"
ENRICHMENT_ALL_OUTPUT = OUTPUT_DIR / "posthoc_enrichment_all_thresholds.tsv"
CLASSIFICATION_COUNTS_ALL_OUTPUT = OUTPUT_DIR / "classification_counts_all_thresholds.tsv"
CLASSIFICATION_PRIMARY_GROUP_ALL_OUTPUT = OUTPUT_DIR / "classification_primary_group_counts_all_thresholds.tsv"
TOP_OUTLIERS_ALL_OUTPUT = OUTPUT_DIR / "top_outliers_all_thresholds.tsv"

# Backward-compatible aliases for the main T95 downstream pipeline.
SCORED_OUTPUT = SCORED_OUTPUTS[MAIN_THRESHOLD_NAME]
THRESHOLD_OUTPUT = THRESHOLD_OUTPUTS[MAIN_THRESHOLD_NAME]
ENRICHMENT_OUTPUT = ENRICHMENT_OUTPUTS[MAIN_THRESHOLD_NAME]
CLASSIFICATION_COUNTS_OUTPUT = CLASSIFICATION_COUNTS_OUTPUTS[MAIN_THRESHOLD_NAME]
CLASSIFICATION_PRIMARY_GROUP_OUTPUT = CLASSIFICATION_PRIMARY_GROUP_OUTPUTS[MAIN_THRESHOLD_NAME]
TOP_OUTLIERS_OUTPUT = TOP_OUTLIERS_OUTPUTS[MAIN_THRESHOLD_NAME]
FIGURE_DIR = FIGURE_DIRS[MAIN_THRESHOLD_NAME]

# ============================================================
# Load model input
# ============================================================

df = pd.read_csv(INPUT_PATH, sep="\t", low_memory=False)

print("Input shape:", df.shape)
print("\nAnalysis group distribution:")
print(df["analysis_group"].value_counts(dropna=False))


In [ ]:
gene_annotation_cols = [
    "variant_id",
    "gene_constraint_symbol",
    "gene_constraint_start_position",
    "gene_constraint_end_position",
    "gene_constraint_consequence",
]

gene_annotation = pd.read_csv(
    MASTER_PATH,
    sep="\t",
    usecols=gene_annotation_cols,
    low_memory=False,
)

gene_annotation = gene_annotation.drop_duplicates("variant_id")

df = df.merge(
    gene_annotation,
    on="variant_id",
    how="left",
    validate="one_to_one",
)

In [ ]:
# ============================================================
# Load UCSC phyloP100way scores
# ============================================================

def load_ucsc_phylop_table(path, score_col):
    """
    Load UCSC phyloP bedGraph-like table.

    Expected format:
        track name="..."
        chrom    start    end    value

    Coordinates:
        start is 0-based
        end is non-inclusive
        mtDNA 1-based position = start + 1
    """
    df = pd.read_csv(path, sep="\t", skiprows=1)

    required_cols = {"chrom", "start", "end", "value"}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(
            f"{path} is missing required columns: {missing_cols}"
        )

    df = df.copy()

    df["chrom"] = df["chrom"].astype(str).str.replace('"', "", regex=False)
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")
    df[score_col] = pd.to_numeric(df["value"], errors="coerce")

    # UCSC/BED-like coordinates:
    # one-base interval [start, end) corresponds to 1-based position start + 1.
    df["position"] = df["start"].astype("Int64") + 1

    df = df[["position", score_col]].copy()

    if df["position"].duplicated().any():
        duplicated_positions = df.loc[
            df["position"].duplicated(),
            "position",
        ].head(10).tolist()

        raise ValueError(
            f"Duplicated positions found in {path}: "
            f"{duplicated_positions}"
        )

    print(f"\nLoaded {score_col}:")
    print(df.shape)
    print(df[score_col].describe())

    return df


phylop100 = load_ucsc_phylop_table(
    PHYLOP100WAY_PATH,
    "phyloP100way",
)


In [ ]:
# ============================================================
# Feature engineering
# ============================================================

work = df.copy()

required_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
    "validation_label",
    "is_neutral_dataset8",
    "is_pathogenic_dataset9",
    "is_disease_suspected_dataset3",
    "analysis_group",

    "mlc_score",
    "pop_af_max",
    "pop_af_hom_max",
    "pop_af_het_max",
]

missing_cols = [col for col in required_cols if col not in work.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

numeric_cols = [
    "mlc_score",
    "pop_af_max",
    "pop_af_hom_max",
    "pop_af_het_max",
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")

missing_summary = work[numeric_cols].isna().sum()
missing_summary = missing_summary[missing_summary > 0]

if len(missing_summary) > 0:
    raise ValueError(f"Missing values in numeric features:\n{missing_summary}")

eps = 1e-6

work["rarity_soft"] = -np.log10(work["pop_af_max"] + eps)
work["rarity_soft"] = work["rarity_soft"].clip(lower=0, upper=6)

work["hom_rarity_soft"] = -np.log10(work["pop_af_hom_max"] + eps)
work["hom_rarity_soft"] = work["hom_rarity_soft"].clip(lower=0, upper=6)

work["het_rarity_soft"] = -np.log10(work["pop_af_het_max"] + eps)
work["het_rarity_soft"] = work["het_rarity_soft"].clip(lower=0, upper=6)

work["no_homoplasmic_signal"] = (
    work["pop_af_hom_max"] == 0
).astype(int)

In [ ]:
# ============================================================
# Merge UCSC phyloP100way scores into work table
# ============================================================

work["position"] = pd.to_numeric(work["position"], errors="coerce").astype("Int64")

work = work.merge(
    phylop100,
    on="position",
    how="left",
)

phylop_cols = [
    "phyloP100way",
]

print("\nMissing phyloP values after merge:")
print(work[phylop_cols].isna().sum())

print("\nphyloP100way summary after merge:")
print(work[phylop_cols].describe())

print("\nphyloP100way by analysis group:")
print(
    work
    .groupby("analysis_group")[phylop_cols]
    .agg(["count", "mean", "median", "std"])
)


In [ ]:
# ============================================================
# CDS-aware codon-position features
# ============================================================
# Previous version used absolute mtDNA coordinate:
#     position % 3
#
# This is not correct for coding interpretation, because protein
# translation starts from the beginning of each gene/CDS, not from
# mtDNA coordinate 1.
#
# New logic:
#     relative_nt_position = position - gene_start + 1
#     codon_position = ((relative_nt_position - 1) % 3) + 1
#
# For MT-ND6, which is encoded on the reverse strand, the coding
# direction is opposite to the reference coordinate direction:
#     relative_nt_position = gene_end - position + 1

work["position_numeric"] = pd.to_numeric(
    work["position"],
    errors="coerce",
)

work["gene_constraint_start_position"] = pd.to_numeric(
    work["gene_constraint_start_position"],
    errors="coerce",
)

work["gene_constraint_end_position"] = pd.to_numeric(
    work["gene_constraint_end_position"],
    errors="coerce",
)

work["gene_constraint_symbol_clean"] = (
    work["gene_constraint_symbol"]
    .astype(str)
    .str.upper()
    .str.replace(" ", "", regex=False)
)

# Protein-coding variants are those with gene-level constraint coordinates.
# Non-coding variants will receive codon_position_simple = "noncoding".
work["is_coding_gene_constraint"] = (
    work["gene_constraint_start_position"].notna()
    & work["gene_constraint_end_position"].notna()
).astype(int)

# Most human mtDNA protein-coding genes are on the forward strand.
# MT-ND6 is the important reverse-strand exception.
reverse_strand_genes = {
    "MT-ND6",
    "ND6",
}

work["is_reverse_strand_gene"] = (
    work["gene_constraint_symbol_clean"].isin(reverse_strand_genes)
).astype(int)

work["gene_relative_nt_position"] = np.nan

forward_mask = (
    (work["is_coding_gene_constraint"] == 1)
    & (work["is_reverse_strand_gene"] == 0)
)

reverse_mask = (
    (work["is_coding_gene_constraint"] == 1)
    & (work["is_reverse_strand_gene"] == 1)
)

# Forward-strand coding genes:
# first coding nucleotide = gene start
work.loc[forward_mask, "gene_relative_nt_position"] = (
    work.loc[forward_mask, "position_numeric"]
    - work.loc[forward_mask, "gene_constraint_start_position"]
    + 1
)

# Reverse-strand coding gene MT-ND6:
# first coding nucleotide is at the higher coordinate end
work.loc[reverse_mask, "gene_relative_nt_position"] = (
    work.loc[reverse_mask, "gene_constraint_end_position"]
    - work.loc[reverse_mask, "position_numeric"]
    + 1
)

# Keep legacy column name for downstream compatibility.
# Now it is gene-frame modulo, not absolute mtDNA-coordinate modulo.
work["position_mod3"] = np.nan

coding_mask = work["gene_relative_nt_position"].notna()

work.loc[coding_mask, "position_mod3"] = (
    work.loc[coding_mask, "gene_relative_nt_position"] % 3
)

work["codon_pos1_any"] = (work["position_mod3"] == 1).astype(int)
work["codon_pos2_any"] = (work["position_mod3"] == 2).astype(int)
work["codon_pos3_any"] = (work["position_mod3"] == 0).astype(int)

work["codon_position_simple"] = np.select(
    [
        work["position_mod3"] == 1,
        work["position_mod3"] == 2,
        work["position_mod3"] == 0,
    ],
    [
        "pos1",
        "pos2",
        "pos3",
    ],
    default="noncoding",
)

print("\nCDS-aware codon-position feature distribution:")
print(work["codon_position_simple"].value_counts(dropna=False))

print("\nCDS-aware codon-position features by analysis group:")
print(
    pd.crosstab(
        work["analysis_group"],
        work["codon_position_simple"],
        dropna=False,
    )
)

In [ ]:
# ============================================================
# Final feature set: CDS-aware codon position + phyloP100way
# ============================================================

final_feature_cols = [
    "mlc_score",
    "rarity_soft",
    "hom_rarity_soft",
    "het_rarity_soft",
    "no_homoplasmic_signal",

    # CDS-aware codon-position features
    "codon_pos1_any",
    "codon_pos2_any",
    "codon_pos3_any",

    # Final evolutionary conservation feature
    "phyloP100way",
]

feature_sets = {
    FINAL_FEATURE_SET_NAME: final_feature_cols,
}

print("\nFinal feature set:")
print(FINAL_FEATURE_SET_NAME)
for col in final_feature_cols:
    print("-", col)

missing_final_features = [
    col for col in final_feature_cols
    if col not in work.columns
]

if missing_final_features:
    raise ValueError(
        f"Missing final feature columns: {missing_final_features}"
    )

missing_values = work[final_feature_cols].isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    raise ValueError(
        "Missing values found in final feature matrix:\n"
        f"{missing_values}"
    )


In [ ]:
# ============================================================
# Split neutral_reference into train and validation
# ============================================================

neutral_df = work[work["analysis_group"] == "neutral_reference"].copy()

print("\nNeutral reference size:", neutral_df.shape)

neutral_train_idx, neutral_val_idx = train_test_split(
    neutral_df.index,
    test_size=0.25,
    random_state=42,
)

print("Neutral train size:", len(neutral_train_idx))
print("Neutral validation size:", len(neutral_val_idx))

work["neutral_split"] = "not_neutral_reference"
work.loc[neutral_train_idx, "neutral_split"] = "neutral_train"
work.loc[neutral_val_idx, "neutral_split"] = "neutral_validation"

print("\nNeutral split distribution:")
print(work["neutral_split"].value_counts())

In [ ]:
# ============================================================
# Isolation Forest helper
# ============================================================

def fit_score_isolation_forest(data, train_idx, feature_cols):
    X_train = data.loc[train_idx, feature_cols]
    X_all = data[feature_cols]

    model = Pipeline([
        ("scaler", RobustScaler()),
        ("model", IsolationForest(
            n_estimators=500,
            contamination="auto",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    model.fit(X_train)

    # decision_function: larger = more normal
    normality_score = model.decision_function(X_all)

    # outlier score: larger = more out-of-neutral-domain
    outlier_score = -normality_score

    return outlier_score, model


In [ ]:
# ============================================================
# Fit final Isolation Forest model
# ============================================================

scored = work.copy()

score_specs = {}

feature_set_name = FINAL_FEATURE_SET_NAME
cols = feature_sets[feature_set_name]

print(f"\nFitting final feature set: {feature_set_name}")
print("Features:", cols)

missing_feature_cols = [col for col in cols if col not in scored.columns]

if missing_feature_cols:
    raise ValueError(
        f"Missing feature columns for {feature_set_name}: "
        f"{missing_feature_cols}"
    )

model_name = FINAL_MODEL_NAME
score_col = f"{model_name}_outlier_score"

iso_score, iso_model = fit_score_isolation_forest(
    scored,
    neutral_train_idx,
    cols,
)

scored[score_col] = iso_score

score_specs[model_name] = {
    "score_col": score_col,
    "model": iso_model,
    "feature_set": feature_set_name,
    "base_model": "isolation_forest",
    "feature_cols": cols,
}

print("\nFitted final model:")
print(
    model_name,
    "| feature_set:",
    score_specs[model_name]["feature_set"],
    "| base_model:",
    score_specs[model_name]["base_model"],
    "| score_col:",
    score_specs[model_name]["score_col"],
)


In [ ]:
# ============================================================
# Threshold calibration on held-out neutral_validation
# One Isolation Forest score, three interpretation thresholds: T90, T95, T99
# ============================================================

threshold_rows = []

validation_mask = scored["neutral_split"] == "neutral_validation"
score_col = score_specs[FINAL_MODEL_NAME]["score_col"]
validation_scores = scored.loc[validation_mask, score_col]

for threshold_name, threshold_quantile in THRESHOLD_CONFIG.items():
    threshold_value = validation_scores.quantile(threshold_quantile)
    flag_col = f"{FINAL_MODEL_NAME}_above_{threshold_name}"

    scored[flag_col] = (
        scored[score_col] > threshold_value
    ).astype(int)

    n_val_above = int(scored.loc[validation_mask, flag_col].sum())
    frac_val_above = float(scored.loc[validation_mask, flag_col].mean())

    threshold_rows.append({
        "model_name": FINAL_MODEL_NAME,
        "feature_set": FINAL_FEATURE_SET_NAME,
        "base_model": "isolation_forest",
        "score_col": score_col,
        "threshold_name": threshold_name,
        "validation_neutral_quantile": threshold_quantile,
        "threshold_value": threshold_value,
        "n_validation_neutral_above": n_val_above,
        "fraction_validation_neutral_above": frac_val_above,
        "expected_validation_neutral_fraction": 1 - threshold_quantile,
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df["final_model_alias"] = "isolation_forest"

print("\nThresholds calibrated on held-out neutral_validation:")
print(threshold_df)


In [ ]:
# ============================================================
# Post-hoc enrichment by analysis group at T90, T95, and T99
# ============================================================

analysis_groups = [
    "neutral_reference",
    "unlabeled_or_other",
    "article_pathogenic_posthoc",
    "disease_suspected_posthoc",
]

enrichment_rows = []

for threshold_name, threshold_quantile in THRESHOLD_CONFIG.items():
    expected_fraction = 1 - threshold_quantile
    flag_col = f"{FINAL_MODEL_NAME}_above_{threshold_name}"

    for group in analysis_groups:
        group_mask = scored["analysis_group"] == group

        n_group = int(group_mask.sum())
        n_above = int(scored.loc[group_mask, flag_col].sum())
        frac_above = n_above / n_group if n_group > 0 else np.nan

        enrichment_rows.append({
            "model_name": FINAL_MODEL_NAME,
            "feature_set": FINAL_FEATURE_SET_NAME,
            "base_model": "isolation_forest",
            "threshold_name": threshold_name,
            "validation_neutral_quantile": threshold_quantile,
            "analysis_group": group,
            "n_group": n_group,
            "n_above_threshold": n_above,
            "n_inside_threshold": n_group - n_above,
            "fraction_above_threshold": frac_above,
            "fraction_inside_threshold": (
                1 - frac_above if pd.notna(frac_above) else np.nan
            ),
            "expected_validation_neutral_fraction": expected_fraction,
            "fold_enrichment_vs_expected": (
                frac_above / expected_fraction
                if expected_fraction > 0 and pd.notna(frac_above)
                else np.nan
            ),
        })

enrichment_df = pd.DataFrame(enrichment_rows)

enrichment_df.to_csv(
    ENRICHMENT_ALL_OUTPUT,
    sep="\t",
    index=False,
)

for threshold_name in THRESHOLD_NAMES:
    enrichment_threshold_df = enrichment_df[
        enrichment_df["threshold_name"] == threshold_name
    ].copy()

    enrichment_threshold_df.to_csv(
        ENRICHMENT_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

print("\nPost-hoc enrichment across thresholds:")
print(enrichment_df)

print("\nSaved combined post-hoc enrichment:")
print(ENRICHMENT_ALL_OUTPUT)

print("\nSaved per-threshold post-hoc enrichment files:")
for threshold_name, output_path in ENRICHMENT_OUTPUTS.items():
    print(threshold_name, "->", output_path)


In [ ]:
# ============================================================
# Save final scored tables for downstream spectrum analysis
# Final model: Isolation Forest + CDS-aware codon position + phyloP100way
# Main downstream threshold: T95
# Sensitivity thresholds: T90 and T99
# ============================================================

# ------------------------------------------------------------
# Check that final model and threshold columns exist
# ------------------------------------------------------------

required_final_model_cols = [
    f"{FINAL_MODEL_NAME}_outlier_score",
]

required_final_model_cols += [
    f"{FINAL_MODEL_NAME}_above_{threshold_name}"
    for threshold_name in THRESHOLD_NAMES
]

missing_final_model_cols = [
    col for col in required_final_model_cols
    if col not in scored.columns
]

if missing_final_model_cols:
    raise ValueError(
        "Final model columns are missing from scored table: "
        f"{missing_final_model_cols}\n"
        "Check that the phyloP100way Isolation Forest model was fitted "
        "and thresholds were calculated."
    )

# ------------------------------------------------------------
# Add rank and percentile for final model
# ------------------------------------------------------------

final_score_col = f"{FINAL_MODEL_NAME}_outlier_score"
final_rank_col = f"{FINAL_MODEL_NAME}_rank"
final_percentile_col = f"{FINAL_MODEL_NAME}_percentile"

scored[final_rank_col] = scored[final_score_col].rank(
    ascending=False,
    method="min",
)

scored[final_percentile_col] = scored[final_score_col].rank(
    pct=True,
    ascending=True,
)

# ------------------------------------------------------------
# Create final scored table with simplified column names
# ------------------------------------------------------------

final_scored = scored.copy()

final_scored["isolation_forest_outlier_score"] = (
    final_scored[f"{FINAL_MODEL_NAME}_outlier_score"]
)

final_scored["isolation_forest_rank"] = (
    final_scored[f"{FINAL_MODEL_NAME}_rank"]
)

final_scored["isolation_forest_percentile"] = (
    final_scored[f"{FINAL_MODEL_NAME}_percentile"]
)

threshold_value_by_name = (
    threshold_df
    .set_index("threshold_name")["threshold_value"]
    .astype(float)
    .to_dict()
)

for threshold_name in THRESHOLD_NAMES:
    flag_col_raw = f"{FINAL_MODEL_NAME}_above_{threshold_name}"
    flag_col_simple = f"isolation_forest_above_{threshold_name}"
    threshold_col_simple = f"isolation_forest_threshold_{threshold_name}"
    status_col = f"threshold_status_{threshold_name}"

    final_scored[flag_col_simple] = final_scored[flag_col_raw]
    final_scored[threshold_col_simple] = threshold_value_by_name[threshold_name]

    final_scored[status_col] = np.where(
        final_scored[flag_col_simple] == 1,
        f"above_{threshold_name}",
        f"inside_{threshold_name}",
    )

# Backward-compatible alias used by old plotting/sanity checks.
final_scored["t95_status"] = final_scored["threshold_status_T95"]


def assign_neutral_domain_class(row, threshold_name):
    group = row["analysis_group"]
    above = row[f"isolation_forest_above_{threshold_name}"] == 1

    if group == "neutral_reference":
        return (
            f"known_neutral_reference_above_{threshold_name}"
            if above
            else f"known_neutral_reference_inside_{threshold_name}"
        )

    if group == "unlabeled_or_other":
        return (
            f"unlabeled_out_of_neutral_domain_{threshold_name}"
            if above
            else f"unlabeled_neutral_like_{threshold_name}"
        )

    if group == "article_pathogenic_posthoc":
        return (
            f"article_pathogenic_above_{threshold_name}"
            if above
            else f"article_pathogenic_inside_{threshold_name}"
        )

    if group == "disease_suspected_posthoc":
        return (
            f"disease_suspected_above_{threshold_name}"
            if above
            else f"disease_suspected_inside_{threshold_name}"
        )

    return "other"


def assign_primary_spectrum_group_preview(row, threshold_name):
    group = row["analysis_group"]
    above = row[f"isolation_forest_above_{threshold_name}"] == 1

    if group == "neutral_reference":
        return f"expanded_neutral_like_{threshold_name}"

    if group == "unlabeled_or_other":
        return (
            f"unlabeled_out_of_neutral_domain_{threshold_name}"
            if above
            else f"expanded_neutral_like_{threshold_name}"
        )

    return "exclude_posthoc_or_other"


for threshold_name in THRESHOLD_NAMES:
    final_scored[f"neutral_domain_class_{threshold_name}"] = final_scored.apply(
        assign_neutral_domain_class,
        axis=1,
        threshold_name=threshold_name,
    )

    final_scored[f"spectrum_group_primary_{threshold_name}_preview"] = final_scored.apply(
        assign_primary_spectrum_group_preview,
        axis=1,
        threshold_name=threshold_name,
    )

# ------------------------------------------------------------
# Output columns
# ------------------------------------------------------------

base_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
    "validation_label",
    "is_neutral_dataset8",
    "is_pathogenic_dataset9",
    "is_disease_suspected_dataset3",
    "analysis_group",
    "neutral_split",
]

annotation_cols = [
    # Supplementary Dataset 3 annotation.
    # Used only for post-hoc interpretation, not as model feature.
    "Classification_group",

    # CDS-aware codon-position annotation
    "position_mod3",
    "codon_position_simple",
]

feature_cols_final = [
    "mlc_score",

    "pop_af_max",
    "pop_af_hom_max",
    "pop_af_het_max",

    "rarity_soft",
    "hom_rarity_soft",
    "het_rarity_soft",
    "no_homoplasmic_signal",

    "codon_pos1_any",
    "codon_pos2_any",
    "codon_pos3_any",

    "phyloP100way",
]

model_base_output_cols = [
    "isolation_forest_outlier_score",
    "isolation_forest_rank",
    "isolation_forest_percentile",
]

threshold_output_cols = []

for threshold_name in THRESHOLD_NAMES:
    threshold_output_cols.extend([
        f"isolation_forest_threshold_{threshold_name}",
        f"isolation_forest_above_{threshold_name}",
        f"threshold_status_{threshold_name}",
        f"neutral_domain_class_{threshold_name}",
        f"spectrum_group_primary_{threshold_name}_preview",
    ])

# Compatibility alias for T95.
threshold_output_cols.append("t95_status")

final_output_cols_all = (
    [col for col in base_cols if col in final_scored.columns]
    + [col for col in annotation_cols if col in final_scored.columns]
    + [col for col in feature_cols_final if col in final_scored.columns]
    + model_base_output_cols
    + threshold_output_cols
)

missing_final_output_cols = [
    col for col in final_output_cols_all
    if col not in final_scored.columns
]

if missing_final_output_cols:
    raise ValueError(
        f"Missing final output columns: {missing_final_output_cols}"
    )

# ------------------------------------------------------------
# Save combined and per-threshold scored tables
# ------------------------------------------------------------

final_scored[final_output_cols_all].to_csv(
    SCORED_ALL_OUTPUT,
    sep="\t",
    index=False,
)

print("\nSaved combined scored table with all thresholds:")
print(SCORED_ALL_OUTPUT)
print("Shape:", final_scored[final_output_cols_all].shape)

threshold_specific_output_cols = {}

for threshold_name in THRESHOLD_NAMES:
    threshold_specific_cols = [
        f"isolation_forest_threshold_{threshold_name}",
        f"isolation_forest_above_{threshold_name}",
        f"threshold_status_{threshold_name}",
        f"neutral_domain_class_{threshold_name}",
        f"spectrum_group_primary_{threshold_name}_preview",
    ]

    if threshold_name == "T95":
        threshold_specific_cols.append("t95_status")

    final_output_cols_threshold = (
        [col for col in base_cols if col in final_scored.columns]
        + [col for col in annotation_cols if col in final_scored.columns]
        + [col for col in feature_cols_final if col in final_scored.columns]
        + model_base_output_cols
        + threshold_specific_cols
    )

    threshold_specific_output_cols[threshold_name] = final_output_cols_threshold

    final_scored[final_output_cols_threshold].to_csv(
        SCORED_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

print("\nSaved per-threshold scored tables:")
for threshold_name, output_path in SCORED_OUTPUTS.items():
    print(threshold_name, "->", output_path)

# ------------------------------------------------------------
# Save threshold tables
# ------------------------------------------------------------

threshold_df.to_csv(
    THRESHOLDS_ALL_OUTPUT,
    sep="\t",
    index=False,
)

for threshold_name in THRESHOLD_NAMES:
    threshold_subset = threshold_df[
        threshold_df["threshold_name"] == threshold_name
    ].copy()

    threshold_subset.to_csv(
        THRESHOLD_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

print("\nSaved combined threshold table:")
print(THRESHOLDS_ALL_OUTPUT)

print("\nSaved per-threshold tables:")
for threshold_name, output_path in THRESHOLD_OUTPUTS.items():
    print(threshold_name, "->", output_path)

print("\nThreshold summary:")
print(threshold_df)


In [ ]:
# ============================================================
# Classification summaries for inspection and plots
# ============================================================

classification_counts_rows = []
primary_group_counts_rows = []

for threshold_name in THRESHOLD_NAMES:
    status_col = f"threshold_status_{threshold_name}"
    primary_group_col = f"spectrum_group_primary_{threshold_name}_preview"

    classification_counts_threshold = (
        final_scored
        .groupby(["analysis_group", status_col], dropna=False)
        .size()
        .reset_index(name="n_variants")
        .rename(columns={status_col: "threshold_status"})
    )

    classification_counts_threshold["threshold_name"] = threshold_name

    classification_totals = (
        final_scored
        .groupby("analysis_group", dropna=False)
        .size()
        .reset_index(name="n_group")
    )

    classification_counts_threshold = classification_counts_threshold.merge(
        classification_totals,
        on="analysis_group",
        how="left",
    )

    classification_counts_threshold["fraction_within_group"] = (
        classification_counts_threshold["n_variants"]
        / classification_counts_threshold["n_group"]
    )

    classification_counts_threshold = classification_counts_threshold.sort_values(
        ["analysis_group", "threshold_status"]
    )

    classification_counts_threshold.to_csv(
        CLASSIFICATION_COUNTS_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

    classification_counts_rows.append(classification_counts_threshold)

    primary_group_counts_threshold = (
        final_scored
        .groupby(primary_group_col, dropna=False)
        .size()
        .reset_index(name="n_variants")
        .rename(columns={primary_group_col: "spectrum_group_primary_preview"})
        .sort_values("n_variants", ascending=False)
    )

    primary_group_counts_threshold["threshold_name"] = threshold_name
    primary_group_counts_threshold["fraction_total"] = (
        primary_group_counts_threshold["n_variants"]
        / primary_group_counts_threshold["n_variants"].sum()
    )

    primary_group_counts_threshold.to_csv(
        CLASSIFICATION_PRIMARY_GROUP_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

    primary_group_counts_rows.append(primary_group_counts_threshold)

classification_counts_all = pd.concat(
    classification_counts_rows,
    ignore_index=True,
)

primary_group_counts_all = pd.concat(
    primary_group_counts_rows,
    ignore_index=True,
)

classification_counts_all.to_csv(
    CLASSIFICATION_COUNTS_ALL_OUTPUT,
    sep="\t",
    index=False,
)

primary_group_counts_all.to_csv(
    CLASSIFICATION_PRIMARY_GROUP_ALL_OUTPUT,
    sep="\t",
    index=False,
)

top_outliers = (
    final_scored[final_output_cols_all]
    .sort_values("isolation_forest_outlier_score", ascending=False)
    .head(100)
)

top_outliers.to_csv(
    TOP_OUTLIERS_ALL_OUTPUT,
    sep="\t",
    index=False,
)

for threshold_name in THRESHOLD_NAMES:
    cols = threshold_specific_output_cols[threshold_name]

    top_outliers[cols].to_csv(
        TOP_OUTLIERS_OUTPUTS[threshold_name],
        sep="\t",
        index=False,
    )

# Backward-compatible aliases for T95 plots below.
classification_counts = classification_counts_all[
    classification_counts_all["threshold_name"] == MAIN_THRESHOLD_NAME
].copy()

primary_group_counts = primary_group_counts_all[
    primary_group_counts_all["threshold_name"] == MAIN_THRESHOLD_NAME
].copy()

t95_enrichment_df = enrichment_df[
    enrichment_df["threshold_name"] == MAIN_THRESHOLD_NAME
].copy()

print("\nClassification counts by analysis group and threshold:")
print(classification_counts_all)

print("\nPrimary spectrum-group preview counts by threshold:")
print(primary_group_counts_all)

print("\nSaved combined classification counts:")
print(CLASSIFICATION_COUNTS_ALL_OUTPUT)

print("\nSaved combined primary group counts:")
print(CLASSIFICATION_PRIMARY_GROUP_ALL_OUTPUT)

print("\nSaved top 100 outliers with all threshold columns:")
print(TOP_OUTLIERS_ALL_OUTPUT)


In [ ]:
# ============================================================
# Visualization functions: final classification at T90/T95/T99
# ============================================================

GROUP_ORDER = [
    "neutral_reference",
    "unlabeled_or_other",
    "article_pathogenic_posthoc",
    "disease_suspected_posthoc",
]

GROUP_LABELS = {
    "neutral_reference": "Neutral reference",
    "unlabeled_or_other": "Unlabeled / other",
    "article_pathogenic_posthoc": "Article pathogenic\npost hoc",
    "disease_suspected_posthoc": "Disease suspected\npost hoc",
}


def save_and_show(save_path=None):
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def get_primary_group_order(threshold_name):
    return [
        f"expanded_neutral_like_{threshold_name}",
        f"unlabeled_out_of_neutral_domain_{threshold_name}",
        "exclude_posthoc_or_other",
    ]


def get_primary_group_labels(threshold_name):
    return {
        f"expanded_neutral_like_{threshold_name}": "Expanded\nneutral-like",
        f"unlabeled_out_of_neutral_domain_{threshold_name}": "Unlabeled\nout-of-neutral-domain",
        "exclude_posthoc_or_other": "Excluded\npost hoc / other",
    }


def plot_counts_by_analysis_group(data, threshold_name, save_path=None):
    status_col = f"threshold_status_{threshold_name}"
    inside_status = f"inside_{threshold_name}"
    above_status = f"above_{threshold_name}"

    status_order = [inside_status, above_status]
    status_labels = {
        inside_status: f"Inside {threshold_name}",
        above_status: f"Above {threshold_name}",
    }

    plot_df = (
        data
        .groupby(["analysis_group", status_col], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reindex(GROUP_ORDER)
        .fillna(0)
    )

    for status in status_order:
        if status not in plot_df.columns:
            plot_df[status] = 0

    plot_df = plot_df[status_order]

    x = np.arange(len(plot_df))
    bottom = np.zeros(len(plot_df))

    plt.figure(figsize=(9, 5))

    for status in status_order:
        values = plot_df[status].values
        plt.bar(
            x,
            values,
            bottom=bottom,
            label=status_labels[status],
        )

        for xi, value, btm in zip(x, values, bottom):
            if value > 0:
                plt.text(
                    xi,
                    btm + value / 2,
                    f"{int(value):,}".replace(",", " "),
                    ha="center",
                    va="center",
                    fontsize=8,
                )

        bottom += values

    plt.xticks(
        x,
        [GROUP_LABELS.get(g, g) for g in plot_df.index],
        rotation=0,
    )
    plt.ylabel("Number of variants")
    plt.title(f"Isolation Forest {threshold_name} classification by analysis group")
    plt.legend(fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)

    return plot_df


def plot_fraction_above_by_analysis_group(enrichment_df, threshold_name, save_path=None):
    plot_df = enrichment_df[
        enrichment_df["threshold_name"] == threshold_name
    ].copy()

    plot_df = plot_df[plot_df["analysis_group"].isin(GROUP_ORDER)].copy()

    plot_df["analysis_group"] = pd.Categorical(
        plot_df["analysis_group"],
        categories=GROUP_ORDER,
        ordered=True,
    )

    plot_df = plot_df.sort_values("analysis_group")

    x = np.arange(plot_df.shape[0])
    y = plot_df["fraction_above_threshold"].values
    expected_fraction = 1 - THRESHOLD_CONFIG[threshold_name]

    plt.figure(figsize=(9, 5))

    plt.bar(x, y)
    plt.axhline(
        expected_fraction,
        linestyle="--",
        linewidth=1,
        label=f"Expected neutral tail at {threshold_name}",
    )

    for xi, yi, n_above, n_group in zip(
        x,
        y,
        plot_df["n_above_threshold"],
        plot_df["n_group"],
    ):
        plt.text(
            xi,
            yi + 0.02,
            f"{int(n_above):,}/{int(n_group):,}".replace(",", " "),
            ha="center",
            va="bottom",
            fontsize=8,
        )

    plt.xticks(
        x,
        [GROUP_LABELS.get(g, str(g)) for g in plot_df["analysis_group"]],
    )
    plt.ylim(0, min(1.05, max(0.2, np.nanmax(y) + 0.15)))
    plt.ylabel(f"Fraction above {threshold_name}")
    plt.title("Fraction of variants outside the neutral-like domain")
    plt.legend(fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)

    return plot_df


def plot_score_distribution_by_analysis_group(data, threshold_value, threshold_name, save_path=None):
    plot_data = []
    labels = []

    for group in GROUP_ORDER:
        values = data.loc[
            data["analysis_group"] == group,
            "isolation_forest_outlier_score",
        ].dropna().values

        if len(values) > 0:
            plot_data.append(values)
            labels.append(GROUP_LABELS.get(group, group))

    plt.figure(figsize=(10, 5))

    plt.boxplot(
        plot_data,
        labels=labels,
        showfliers=False,
        patch_artist=True,
    )

    plt.axhline(
        threshold_value,
        linestyle="--",
        linewidth=1,
        label=f"{threshold_name} threshold",
    )

    plt.ylabel("Isolation Forest outlier score")
    plt.title(f"Outlier-score distribution by analysis group ({threshold_name})")
    plt.xticks(rotation=0)
    plt.legend(fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)


def plot_primary_group_preview_counts(primary_group_counts_all, threshold_name, save_path=None):
    plot_df = primary_group_counts_all[
        primary_group_counts_all["threshold_name"] == threshold_name
    ].copy()

    primary_group_order = get_primary_group_order(threshold_name)
    primary_group_labels = get_primary_group_labels(threshold_name)

    plot_df["spectrum_group_primary_preview"] = pd.Categorical(
        plot_df["spectrum_group_primary_preview"],
        categories=primary_group_order,
        ordered=True,
    )

    plot_df = (
        plot_df
        .sort_values("spectrum_group_primary_preview")
        .dropna(subset=["spectrum_group_primary_preview"])
    )

    x = np.arange(plot_df.shape[0])
    y = plot_df["n_variants"].values

    plt.figure(figsize=(8, 5))

    plt.bar(x, y)

    for xi, yi, frac in zip(
        x,
        y,
        plot_df["fraction_total"].values,
    ):
        plt.text(
            xi,
            yi,
            f"{int(yi):,}\n({frac:.1%})".replace(",", " "),
            ha="center",
            va="bottom",
            fontsize=8,
        )

    plt.xticks(
        x,
        [
            primary_group_labels.get(g, str(g))
            for g in plot_df["spectrum_group_primary_preview"]
        ],
    )

    plt.ylabel("Number of variants")
    plt.title(f"Primary grouping preview after {threshold_name} classification")
    plt.tight_layout()

    save_and_show(save_path)

    return plot_df


def plot_codon_position_by_classification(data, threshold_name, save_path=None):
    flag_col = f"isolation_forest_above_{threshold_name}"

    required_cols = {
        "analysis_group",
        "codon_position_simple",
        flag_col,
    }

    missing_cols = required_cols - set(data.columns)

    if missing_cols:
        raise ValueError(
            f"Cannot plot codon-position heatmap. Missing columns: {missing_cols}"
        )

    plot_df = data[data["analysis_group"] == "unlabeled_or_other"].copy()

    plot_df["classification"] = np.where(
        plot_df[flag_col] == 1,
        f"out_of_neutral_domain_{threshold_name}",
        f"neutral_like_{threshold_name}",
    )

    codon_order = [
        "pos1",
        "pos2",
        "pos3",
        "noncoding",
    ]

    class_order = [
        f"neutral_like_{threshold_name}",
        f"out_of_neutral_domain_{threshold_name}",
    ]

    pivot_df = (
        plot_df
        .groupby(["codon_position_simple", "classification"], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reindex(codon_order)
        .fillna(0)
    )

    for col in class_order:
        if col not in pivot_df.columns:
            pivot_df[col] = 0

    pivot_df = pivot_df[class_order]

    matrix = pivot_df.values

    plt.figure(figsize=(7, 4))
    plt.imshow(matrix, aspect="auto")

    plt.xticks(
        np.arange(len(class_order)),
        ["Neutral-like", "Out-of-neutral-domain"],
    )
    plt.yticks(
        np.arange(len(codon_order)),
        codon_order,
    )

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(
                j,
                i,
                f"{int(matrix[i, j]):,}".replace(",", " "),
                ha="center",
                va="center",
                fontsize=9,
            )

    plt.colorbar(label="Number of unlabeled variants")
    plt.title(f"Unlabeled variants by codon position and {threshold_name} class")
    plt.tight_layout()

    save_and_show(save_path)

    return pivot_df


def plot_fraction_above_across_thresholds(enrichment_df, save_path=None):
    plot_df = enrichment_df[
        enrichment_df["analysis_group"].isin(GROUP_ORDER)
    ].copy()

    pivot_df = (
        plot_df
        .pivot(
            index="analysis_group",
            columns="threshold_name",
            values="fraction_above_threshold",
        )
        .reindex(GROUP_ORDER)
    )

    threshold_order = THRESHOLD_NAMES
    pivot_df = pivot_df[threshold_order]

    x = np.arange(len(pivot_df.index))
    width = 0.22

    plt.figure(figsize=(10, 5))

    for i, threshold_name in enumerate(threshold_order):
        offset = (i - (len(threshold_order) - 1) / 2) * width
        values = pivot_df[threshold_name].values
        plt.bar(
            x + offset,
            values,
            width=width,
            label=threshold_name,
        )

    plt.xticks(
        x,
        [GROUP_LABELS.get(g, g) for g in pivot_df.index],
    )
    plt.ylabel("Fraction above threshold")
    plt.title("Sensitivity of out-of-neutral-domain calls to threshold choice")
    plt.legend(title="Threshold", fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)

    return pivot_df


def plot_primary_group_counts_across_thresholds(primary_group_counts_all, save_path=None):
    plot_df = primary_group_counts_all.copy()

    records = []

    for threshold_name in THRESHOLD_NAMES:
        neutral_like = f"expanded_neutral_like_{threshold_name}"
        out_domain = f"unlabeled_out_of_neutral_domain_{threshold_name}"

        subset = plot_df[plot_df["threshold_name"] == threshold_name]
        counts = dict(
            zip(
                subset["spectrum_group_primary_preview"],
                subset["n_variants"],
            )
        )

        records.append({
            "threshold_name": threshold_name,
            "expanded_neutral_like": counts.get(neutral_like, 0),
            "unlabeled_out_of_neutral_domain": counts.get(out_domain, 0),
            "exclude_posthoc_or_other": counts.get("exclude_posthoc_or_other", 0),
        })

    pivot_df = pd.DataFrame(records).set_index("threshold_name")

    group_cols = [
        "expanded_neutral_like",
        "unlabeled_out_of_neutral_domain",
        "exclude_posthoc_or_other",
    ]

    labels = {
        "expanded_neutral_like": "Expanded neutral-like",
        "unlabeled_out_of_neutral_domain": "Unlabeled out-of-neutral-domain",
        "exclude_posthoc_or_other": "Excluded post hoc / other",
    }

    x = np.arange(len(pivot_df.index))
    width = 0.25

    plt.figure(figsize=(9, 5))

    for i, col in enumerate(group_cols):
        offset = (i - (len(group_cols) - 1) / 2) * width
        values = pivot_df[col].values
        plt.bar(
            x + offset,
            values,
            width=width,
            label=labels[col],
        )

        for xi, yi in zip(x + offset, values):
            if yi > 0:
                plt.text(
                    xi,
                    yi,
                    f"{int(yi):,}".replace(",", " "),
                    ha="center",
                    va="bottom",
                    fontsize=7,
                    rotation=90,
                )

    plt.xticks(x, pivot_df.index)
    plt.ylabel("Number of variants")
    plt.title("Primary group sizes across T90, T95, and T99")
    plt.legend(fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)

    return pivot_df


def plot_score_distribution_with_all_thresholds(data, threshold_df, save_path=None):
    plot_data = []
    labels = []

    for group in GROUP_ORDER:
        values = data.loc[
            data["analysis_group"] == group,
            "isolation_forest_outlier_score",
        ].dropna().values

        if len(values) > 0:
            plot_data.append(values)
            labels.append(GROUP_LABELS.get(group, group))

    plt.figure(figsize=(10, 5))

    plt.boxplot(
        plot_data,
        labels=labels,
        showfliers=False,
        patch_artist=True,
    )

    for _, row in threshold_df.iterrows():
        plt.axhline(
            float(row["threshold_value"]),
            linestyle="--",
            linewidth=1,
            label=row["threshold_name"],
        )

    plt.ylabel("Isolation Forest outlier score")
    plt.title("Outlier-score distribution with T90, T95, and T99 thresholds")
    plt.xticks(rotation=0)
    plt.legend(title="Threshold", fontsize=8)
    plt.tight_layout()

    save_and_show(save_path)


In [ ]:
# ============================================================
# Generate classification plots for T90, T95, and T99
# ============================================================

plot_tables_by_threshold = {}

for threshold_name in THRESHOLD_NAMES:
    threshold_value = threshold_value_by_name[threshold_name]
    figure_dir = FIGURE_DIRS[threshold_name]

    print(f"\nGenerating plots for {threshold_name} -> {figure_dir}")

    counts_plot_df = plot_counts_by_analysis_group(
        data=final_scored,
        threshold_name=threshold_name,
        save_path=figure_dir / f"classification_counts_by_analysis_group_{threshold_name}.png",
    )

    fraction_plot_df = plot_fraction_above_by_analysis_group(
        enrichment_df=enrichment_df,
        threshold_name=threshold_name,
        save_path=figure_dir / f"classification_fraction_above_{threshold_name}.png",
    )

    plot_score_distribution_by_analysis_group(
        data=final_scored,
        threshold_value=threshold_value,
        threshold_name=threshold_name,
        save_path=figure_dir / f"score_distribution_by_analysis_group_{threshold_name}.png",
    )

    primary_plot_df = plot_primary_group_preview_counts(
        primary_group_counts_all=primary_group_counts_all,
        threshold_name=threshold_name,
        save_path=figure_dir / f"primary_group_preview_counts_{threshold_name}.png",
    )

    codon_heatmap_df = plot_codon_position_by_classification(
        data=final_scored,
        threshold_name=threshold_name,
        save_path=figure_dir / f"codon_position_by_classification_{threshold_name}.png",
    )

    plot_tables_by_threshold[threshold_name] = {
        "counts_plot_df": counts_plot_df,
        "fraction_plot_df": fraction_plot_df,
        "primary_plot_df": primary_plot_df,
        "codon_heatmap_df": codon_heatmap_df,
    }

# ------------------------------------------------------------
# Combined sensitivity plots across thresholds
# ------------------------------------------------------------

fraction_sensitivity_df = plot_fraction_above_across_thresholds(
    enrichment_df=enrichment_df,
    save_path=SENSITIVITY_FIGURE_DIR / "classification_fraction_above_T90_T95_T99.png",
)

primary_sensitivity_df = plot_primary_group_counts_across_thresholds(
    primary_group_counts_all=primary_group_counts_all,
    save_path=SENSITIVITY_FIGURE_DIR / "primary_group_counts_T90_T95_T99.png",
)

plot_score_distribution_with_all_thresholds(
    data=final_scored,
    threshold_df=threshold_df,
    save_path=SENSITIVITY_FIGURE_DIR / "score_distribution_with_T90_T95_T99.png",
)

print("\nThreshold summary:")
print(threshold_df)

print("\nFraction-above-threshold sensitivity table:")
print(fraction_sensitivity_df)

print("\nPrimary group-count sensitivity table:")
print(primary_sensitivity_df)

print("\nT95 tables retained for downstream interpretation:")
print("Classification counts:")
print(classification_counts)
print("\nPrimary grouping preview:")
print(primary_group_counts)
